In [ ]:
!mkdir -p /data/sets/nuscenes  # Make the directory to store the nuScenes dataset in.

In [ ]:
!wget https://www.nuscenes.org/data/v1.0-mini.tgz  # Download the nuScenes mini split.

In [ ]:
!tar -xf v1.0-mini.tgz -C /data/sets/nuscenes  # Uncompress the nuScenes mini split.

In [ ]:
!pip uninstall -y numpy scipy scikit-learn
!pip install numpy==1.26.4 scipy==1.11.4 scikit-learn==1.3.2        # Install compatible numpy, scipy, and scikit-learn

In [ ]:
import os
os.kill(os.getpid(), 9)     # Restart runtime

In [ ]:
!pip install nuscenes-devkit        # Install nuscenes-devkit

In [ ]:
!pip install nuscenes-devkit ultralytics -q     # Install nuScenes-devkit ultralytics

In [ ]:
# nuScenes-devkit ultralytics check
from nuscenes.nuscenes import NuScenes
from ultralytics import YOLO
import numpy as np
print("✅ nuscenes-devkit OK")
print("✅ ultralytics OK")

In [ ]:
# Connect google drive 
from google.colab import drive
drive.mount('/content/drive')

NUSCENES_ROOT = '/data/sets/nuscenes'

import os
expected = ['samples', 'sweeps', 'maps', 'v1.0-mini']
for folder in expected:
  path = os.path.join(NUSCENES_ROOT, folder)
  status = "✅" if os.path.exists(path) else "missing"
  print(f"{status} {folder}/")

In [ ]:
nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=True)
my_sammple = nusc.sample[0]
nusc.render_sample(my_sammple['token'])         # NuScenes render_sample

In [ ]:
from PIL import Image
cam_token = my_sample['data']['CAM_FRONT']
cam_data = nusc.get('sample_data', cam_token)
img_path = os.path.join(NUSCENES_ROOT, cam_data['filename'])
print(f"Image path: {img_path}")
img = Image.open(img_path)
model = YOLO("yolo26n.pt")
results = model(img_path)
results[0].show()
print(f"✅ YOLO26 detected {len(results[0].boxes)} objects")        # yolo26n.pt model object detection on NuScenes sample 'CAM_FRONT' image test

In [ ]:
OUTPUT_DIR = '/content/nuscenes_yolo'

In [ ]:
# Create yaml file
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val

names:
  0: car
  1: pedestrian
  2: bicycle
  3: motorcycle
  4: bus
  5: truck
  6: traffic_cone
  7: barrier
"""

yaml_path = f'{OUTPUT_DIR}/nuscenes.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(open(yaml_path).read())
print(f"✅ Saved to {yaml_path}")

In [ ]:
# Nuscenes to YOLO Detection convertion
import os
import numpy as np
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.geometry_utils import view_points, BoxVisibility
from nuscenes.utils.data_classes import Box
from pyquaternion import Quaternion
from pathlib import Path
import shutil
import random

NUSCENES_ROOT = '/data/sets/nuscenes'
OUTPUT_DIR = '/content/nuscenes_yolo'

CLASS_MAP = {
    'car': 0, 'pedestrian': 1, 'bicycle': 2, 'motorcycle': 3,
    'bus': 4, 'truck': 5, 'traffic_cone': 6, 'barrier': 7,
}

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)

# Fresh output folders
if Path(OUTPUT_DIR).exists():
    shutil.rmtree(OUTPUT_DIR)
for split in ['train', 'val']:
    Path(f'{OUTPUT_DIR}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{OUTPUT_DIR}/labels/{split}').mkdir(parents=True, exist_ok=True)

# Collect all samples
all_samples = []
for scene in nusc.scene:
    token = scene['first_sample_token']
    while token:
        all_samples.append(nusc.get('sample', token))
        token = nusc.get('sample', token)['next']

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.8)
splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}
print(f"Total: {len(all_samples)} → train: {split_idx}, val: {len(all_samples)-split_idx}")

converted, total_boxes = 0, 0

for split, samples in splits.items():
    for sample in samples:
        cam_token = sample['data']['CAM_FRONT']
        cam_data  = nusc.get('sample_data', cam_token)
        img_path  = os.path.join(NUSCENES_ROOT, cam_data['filename'])
        img_w, img_h = cam_data['width'], cam_data['height']

        # Camera calibration
        cs  = nusc.get('calibrated_sensor', cam_data['calibrated_sensor_token'])
        ep  = nusc.get('ego_pose', cam_data['ego_pose_token'])
        intrinsic = np.array(cs['camera_intrinsic'])

        # Get boxes in global frame, transform to camera frame
        boxes = nusc.get_boxes(cam_token)
        labels = []

        for box in boxes:
            # Map class
            cat_simple = next((k for k in CLASS_MAP if k in box.name), None)
            if cat_simple is None:
                continue

            # 1. Global → ego vehicle frame
            box.translate(-np.array(ep['translation']))
            box.rotate(Quaternion(ep['rotation']).inverse)

            # 2. Ego → camera frame
            box.translate(-np.array(cs['translation']))
            box.rotate(Quaternion(cs['rotation']).inverse)

            # 3. Skip if behind camera (z < 0.1 means too close or behind)
            if box.center[2] < 0.1:
                continue

            # 4. Project 3D corners to 2D image plane
            corners = view_points(box.corners(), intrinsic, normalize=True)
            xs, ys = corners[0], corners[1]

            x1, x2 = xs.min(), xs.max()
            y1, y2 = ys.min(), ys.max()

            # 5. Skip if completely outside image
            if x2 < 0 or x1 > img_w or y2 < 0 or y1 > img_h:
                continue

            # 6. Clip to image bounds
            x1 = np.clip(x1, 0, img_w)
            x2 = np.clip(x2, 0, img_w)
            y1 = np.clip(y1, 0, img_h)
            y2 = np.clip(y2, 0, img_h)

            # 7. Skip tiny or degenerate boxes
            if x2 - x1 < 4 or y2 - y1 < 4:
                continue
            if (x2 - x1) / img_w > 0.95 or (y2 - y1) / img_h > 0.95:
                continue

            # 8. YOLO format (normalized center + size)
            cx = (x1 + x2) / 2 / img_w
            cy = (y1 + y2) / 2 / img_h
            w  = (x2 - x1) / img_w
            h  = (y2 - y1) / img_h

            labels.append(f"{CLASS_MAP[cat_simple]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            total_boxes += 1

        # Save symlink + label
        img_name = Path(img_path).name
        dst = f'{OUTPUT_DIR}/images/{split}/{img_name}'
        if not os.path.exists(dst):
            os.symlink(img_path, dst)

        with open(f'{OUTPUT_DIR}/labels/{split}/{Path(img_name).stem}.txt', 'w') as f:
            f.write('\n'.join(labels))

        converted += 1

print(f"✅ Done: {converted} images, {total_boxes} total boxes")

# Verify
for split in ['train', 'val']:
    files = list(Path(f'{OUTPUT_DIR}/labels/{split}').glob('*.txt'))
    non_empty = sum(1 for f in files if f.stat().st_size > 0)
    print(f"  {split}: {len(files)} files, {non_empty} non-empty")

# Spot check — should see reasonable values (not 1.0)
from pathlib import Path
sample = next(f for f in Path(f'{OUTPUT_DIR}/labels/train').glob('*.txt') if f.stat().st_size > 0)
print(f"\nSample label ({sample.name}):")
print(open(sample).read()[:300])

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/nuscenes_yolo'
RUNS_DIR = '/content/drive/MyDrive/yolo_runs'       # Set MyDrive as trained model save path

In [ ]:
# Train YOLO26 Detection Model
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo26n_nuscenes",
    project=RUNS_DIR,
    device=0,
    workers=2,
    patience=20,
    save=True,
    plots=True,
)